# Autoregressive generation (PixelCNN, WaveNet) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## factorized sequential likelihood
Probability chain rules are the foundation; convolutions provide masked context in PixelCNN and causal context in WaveNet. This sequential factorization complements diffusion and flows by giving exact likelihood but slower sampling.

In [ ]:
# 1) multiply conditionals into a joint probability
p=np.array([.5,.7,.2]); joint=np.prod(p)
plt.figure(figsize=(4,3)); plt.bar(['p1','p2','p3','joint'],[p[0],p[1],p[2],joint]); plt.title('autoregressive factorization')
assert abs(joint-0.07)<1e-12

In [ ]:
# 2) negative log-likelihood adds token costs
cost=-np.log(p); plt.figure(figsize=(4,3)); plt.bar(range(3),cost); plt.title('sequence NLL is sum of token NLLs')
assert abs(cost.sum()-2.659260)<1e-5

In [ ]:
# 3) masked context forbids future access
mask=np.tril(np.ones((5,5)),k=-1); plt.figure(figsize=(4,4)); plt.imshow(mask,cmap='Greys'); plt.title('causal mask')
assert mask[0,-1]==0 and mask[-1,0]==1

In [ ]:
# 4) sampling proceeds one step at a time
rng=np.random.default_rng(0); probs=[.3,.8,.4,.6]; seq=[]
for pp in probs: seq.append(int(rng.random()<pp))
plt.figure(figsize=(5,3)); plt.step(range(len(seq)),seq,where='mid'); plt.ylim(-.1,1.1); plt.title('sequential sampled bits')
assert len(seq)==4

In [ ]:
# 5) ordering changes which dependency is easy
x=np.array([0,1,0,0]); order1=x; order2=x[::-1]
plt.figure(figsize=(5,3)); plt.plot(order1,'o-',label='forward'); plt.plot(order2,'s-',label='reverse'); plt.legend(); plt.title('different generation orderings')
assert not np.array_equal(order1,order2)